
# SCB Bronze Layer, Workforce Data

Ingests raw JSON from the SCB PxWebApi v2 into an external volume, partitioned by ingestion date.

**API**: `https://api.scb.se/OV0104/v1/doris/en/ssd` (PxWebApi v2)  
**Target**: `/Volumes/labour_market_platform/bronze_work_scb/landing/scb/` (external volume on ADLS)

| Subfolder | Output File(s) | SCB Endpoint | Description |
|---|---|---|---|
| `wages/` | `wages_2016_2022.json`, `wages_2023_2024.json` | `AM/AM0110/AM0110A/LoneSpridSektorYrk4A`, `LoneSpridSektYrk4AN` | Wage distribution by sector, SSYK 2012 occupation |
| `aku/` | `aku_employment_stock_occupation.json` | `AM/AM0401/AM0401I/NAKUSysselYrke2012Ar` | Employment stock by occupation (AKU) |
| `aku/` | `aku_population_region_labourstatus.json` | `AM/AM0401/AM0401N/NAKUBefolkningLAr` | Population by region and labour status (AKU) |
| `aku/` | `aku_unemployed_age_sex.json` | `AM/AM0401/AM0401L/NAKUArblheltidstudAr` | Unemployment by age and sex (AKU) |
| `reference/` | `ssyk_2012_mapping.json` | `AM/AM0110/AM0110A/LoneSpridSektorYrk4A` (Swedish) | SSYK 2012 occupation code metadata |

**Ingestion strategy**:
* Bulk POST for all years per endpoint; falls back to per-year fetch on 403
* Rate-limit handling with incremental backoff on 429
* Partitioned by `ingestion_date=YYYY-MM-DD` for full history

In [0]:
# Configuration
CATALOG    = "labour_market_platform"
SCHEMA     = "bronze_work_scb"
VOLUME     = "landing"

SCB_BASE_URL   = "https://api.scb.se/OV0104/v1/doris/en/ssd"
BRONZE_ROOT    = "scb"
BASE_PATH      = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

# Imports
import requests
import time
import json
import os
from datetime import datetime, UTC
 
INGESTION_DATE = datetime.now(UTC).strftime("%Y-%m-%d")
print(f"Ingestion date: {INGESTION_DATE}")
print(f"Base path:      {BASE_PATH}/{BRONZE_ROOT}/")

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS labour_market_platform.bronze_work_scb;

-- Writes directly to ADLS blob storage
CREATE EXTERNAL VOLUME IF NOT EXISTS labour_market_platform.bronze_work_scb.landing
  LOCATION 'abfss://landing@adlmlabourmarket.dfs.core.windows.net/raw';

In [0]:
def fetch_scb_metadata(endpoint_path: str) -> dict:
    """Fetches variable metadata for a given SCB endpoint.
    Used to discover available years and variable codes before querying."""
 
    response = requests.get(f"{SCB_BASE_URL}/{endpoint_path}")
    response.raise_for_status()
    return response.json()
 
 
def build_scb_query(metadata: dict, years: list) -> dict:
    """Builds the POST query body for a SCB API request.
    Selects all values for every variable, filtered to the requested years for Tid."""
 
    query_items = []
    for variable in metadata["variables"]:
        code = variable["code"]
        if code == "Tid":
            query_items.append({"code": "Tid", "selection": {"filter": "item", "values": years}})
        else:
            query_items.append({"code": code, "selection": {"filter": "item", "values": variable["values"]}})
 
    return {"query": query_items, "response": {"format": "json"}}
 
 
def post_scb_query(endpoint_path: str, query_body: dict) -> dict:
    """Posts a query to the SCB API and returns the raw JSON response."""
 
    response = requests.post(f"{SCB_BASE_URL}/{endpoint_path}", json=query_body)
    response.raise_for_status()
    return response.json()

In [0]:
def write_json_to_bronze(raw_json: dict, subfolder: str, filename: str):
    """Writes a raw JSON response from SCB to the bronze layer.
    Partitioned by ingestion date to preserve a full history across runs."""

    dir_path  = f"{BASE_PATH}/{BRONZE_ROOT}/{subfolder}/ingestion_date={INGESTION_DATE}"
    file_path = f"{dir_path}/{filename}"

    os.makedirs(dir_path, exist_ok=True)
    with open(file_path, "w") as f:
        json.dump(raw_json, f, ensure_ascii=False)

    print(f"  ✓ {file_path}")

In [0]:
def ingest_scb_dataset(endpoint_path: str, years: list, subfolder: str, filename: str):
    """Fetches a SCB dataset and writes the raw JSON response to the bronze layer.
 
    Attempts a bulk fetch for all years first. If the API rejects it, falls back to fetching one year at a time and
    merges the data arrays before writing a single file.
 
    Retries on 429 rate limit responses with incremental backoff."""
 
    print(f"\n--- {filename} ---")
 
    metadata        = fetch_scb_metadata(endpoint_path)
    available_years = next(v["values"] for v in metadata["variables"] if v["code"] == "Tid")
    years_to_fetch  = [y for y in years if y in available_years]
 
    if not years_to_fetch:
        print(f"  No matching years. Available: {available_years}")
        return
 
    print(f"  Years: {years_to_fetch}")
 
    # Bulk fetch
    try:
        query    = build_scb_query(metadata, years_to_fetch)
        raw_json = post_scb_query(endpoint_path, query)
        write_json_to_bronze(raw_json, subfolder, filename)
        time.sleep(2)
        return
 
    except requests.exceptions.HTTPError as e:
        print(f"  Bulk failed ({e.response.status_code}), falling back to per-year fetch...")
 
    # Per-year fallback merges all responses into one JSON file
    merged_data = []
    columns     = None
 
    for year in years_to_fetch:
        for attempt in range(3):
            try:
                time.sleep(3)
                query    = build_scb_query(metadata, [year])
                raw_json = post_scb_query(endpoint_path, query)
 
                merged_data.extend(raw_json.get("data", []))
                if columns is None:
                    columns = raw_json.get("columns")
 
                print(f"  {year}: {len(raw_json.get('data', []))} rows")
                break
 
            except requests.exceptions.HTTPError as e:
                status_code = e.response.status_code
                if status_code == 429:
                    wait_seconds = 15 * (attempt + 1)
                    print(f"  {year}: rate limited, retrying in {wait_seconds}s...")
                    time.sleep(wait_seconds)
                else:
                    print(f"  {year}: failed ({status_code}), skipping.")
                    break
 
    if merged_data:
        combined_json = {"columns": columns, "data": merged_data}
        write_json_to_bronze(combined_json, subfolder, filename)
        print(f"  Total rows: {len(merged_data)}")

In [0]:
# Wages split across two SCB table versions (API schema change at 2023)
ingest_scb_dataset("AM/AM0110/AM0110A/LoneSpridSektorYrk4A", [str(y) for y in range(2016, 2023)], "wages", "wages_2016_2022.json")
ingest_scb_dataset("AM/AM0110/AM0110A/LoneSpridSektYrk4AN",  ["2023", "2024"],                     "wages", "wages_2023_2024.json")

In [0]:
# AKU (Arbetskraftsundersökningen) — employment, population, and unemployment
y_aku = [str(y) for y in range(2016, 2026)]
 
ingest_scb_dataset("AM/AM0401/AM0401I/NAKUSysselYrke2012Ar", y_aku, "aku", "aku_employment_stock_occupation.json")
ingest_scb_dataset("AM/AM0401/AM0401N/NAKUBefolkningLAr",    y_aku, "aku", "aku_population_region_labourstatus.json")
ingest_scb_dataset("AM/AM0401/AM0401L/NAKUArblheltidstudAr", y_aku, "aku", "aku_unemployed_age_sex.json")

In [0]:
# SSYK 2012 occupation code metadata (Swedish endpoint for Swedish occupation names)
SSYK_METADATA_URL = "https://api.scb.se/OV0104/v1/doris/sv/ssd/AM/AM0110/AM0110A/LoneSpridSektorYrk4A"

print("\n--- ssyk_2012_mapping.json ---")
response = requests.get(SSYK_METADATA_URL)
response.raise_for_status()
ssyk_metadata = response.json()

write_json_to_bronze(ssyk_metadata, "reference", "ssyk_2012_mapping.json")

In [0]:
print("\n--- Bronze ingestion complete ---")
print(f"Path: {BASE_PATH}/{BRONZE_ROOT}/")
print(f"Partition: ingestion_date={INGESTION_DATE}")